# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using QAI Hub

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using **Qualcomm AI Hub (QAI Hub)** — Qualcomm's cloud-based service for compiling, quantizing, profiling, and deploying neural network models on real Snapdragon hardware.

Note: Unlike SNPE and QAIRT, which require a local SDK installation, QAI Hub offloads all compilation, quantization, and profiling steps to Qualcomm's cloud infrastructure. Models are submitted as jobs and run on real Snapdragon devices remotely, with results downloadable locally.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **QAI Hub configuration** — authenticate with the QAI Hub API and select the target device.
5. **DLC compilation** — upload the ONNX model to QAI Hub and compile it to QNN DLC format targeting a Snapdragon device.
6. **FP32 profiling** — measure on-device inference latency of the floating-point model.
7. **INT8 quantization** — apply cloud-side post-training quantization using the calibration dataset, then recompile.
8. **INT8 profiling** — measure on-device inference latency of the quantized model across all compute units.

We begin by importing the necessary libraries.

In [ ]:
# Import necessary libraries.
import cv2
import glob
import os
import onnx
import random
import subprocess
import torch
import uuid

import numpy as np
import pandas as pd
import qai_hub as hub

from dotenv import dotenv_values
from libreyolo.models.yolox.nn import LibreYOLOXModel
from onnx import helper, TensorProto
from typing import Tuple

## Preprocessing and Calibration Data

Before uploading or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (letterbox-resized to 640×640, BGR color, 0–255 float32, CHW layout).
- A **representative calibration dataset** in `.raw` binary format. QAI Hub's cloud quantizer (`client.submit_quantize_job()`) accepts calibration inputs as NumPy arrays loaded from these files, and uses them during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `letterbox()` and `preprocess()` helper functions used throughout this pipeline.

In [ ]:
def letterbox(
    image: np.ndarray, target: int = 640, pad_value: int = 114
) -> Tuple:
    """Resize image to (target×target) with top-left letterbox padding.

    YOLOX preprocessing: image is placed at top-left (0, 0); padding fills
    the right and bottom edges. Color format stays BGR, values stay 0–255.

    Args:
        image (np.ndarray): Input BGR image as NumPy array.
        target (int): Target image size (square).
        pad_value (int): Padding pixel value (default 114, YOLOX convention).

    Returns:
        padded (np.ndarray): Padded image, float32 HWC BGR, 0-255 range.
        ratio (float): Scale factor applied to original dimensions.
    """
    h, w = image.shape[:2]
    ratio = min(target / h, target / w)
    new_w, new_h = int(w * ratio), int(h * ratio)

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    padded = np.full((target, target, 3), pad_value, dtype=np.float32)
    padded[:new_h, :new_w] = resized

    return padded, ratio


def preprocess(original_image: np.ndarray, size: int = 640) -> Tuple:
    """
    Preprocess the input image for model inference.

    YOLOX expects: BGR, 0-255 float32, CHW, top-left letterbox.
    No /255 normalisation and no BGR→RGB conversion.

    Args:
        original_image (np.ndarray): Input BGR image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image, CHW layout, 0-255 range.
        float: Scale ratio applied to original dimensions.
    """
    padded, ratio = letterbox(original_image, size)
    return padded.transpose(2, 0, 1), ratio

### Preparing the Calibration Dataset

QAI Hub's cloud quantization workflow (`client.submit_quantize_job()`) accepts calibration inputs as a dictionary of NumPy arrays keyed by input tensor name. These arrays are loaded from `.raw` binary files — flat binary files containing `float32` pixel values with shape `(C, H, W)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB) and **COCO 2017 validation annotation** (~241 MB)
2. Randomly samples **1000 images** for calibration and **100 images** for validation (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files.
5. Generates a `metadata.csv` for data analysis.

In [ ]:
if not os.path.exists("dataset"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d dataset'
    !bash -c 'rm val2017.zip'

    !bash -c 'curl -L -o annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
    !bash -c 'unzip annotations_trainval2017.zip -d dataset'
    !bash -c 'rm annotations_trainval2017.zip'

if not os.path.exists("qaihub/calib"):
    os.makedirs("qaihub/calib", exist_ok=True)
    os.makedirs("qaihub/val", exist_ok=True)

    random.seed(42)
    SAMPLE_SIZE = 1000

    filenames = glob.glob(f"dataset/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        original_image = cv2.imread(filename)
        processed_image, _ = preprocess(original_image)
        processed_image = processed_image.reshape(1, 3, 640, 640)
        processed_image.tofile(f"qaihub/calib/{uuid.uuid4()}.raw")

    !bash -c 'find qaihub/calib -name "*.raw" > qaihub/calib/filenames.txt'

    metadata = []
    for i, filename in enumerate(
        filenames[SAMPLE_SIZE:SAMPLE_SIZE + int(SAMPLE_SIZE * .1)]
    ):
        original_image = cv2.imread(filename)
        processed_image, ratio = preprocess(original_image)
        processed_image = processed_image.reshape(1, 3, 640, 640)
        processed_image.tofile(raw_path := f"qaihub/val/{uuid.uuid4()}.raw")

        metadata.append({
            "raw_path": str(raw_path),
            "image_path": str(filename),
            "image_id": i + 1,
            "orig_w": original_image.shape[1],
            "orig_h": original_image.shape[0],
            "ratio": ratio,
        })

    metadata_df = pd.DataFrame(metadata)
    metadata_df.to_csv("qaihub/val/metadata.csv", index=False)

    !bash -c 'find qaihub/val -name "*.raw" > qaihub/val/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

QAI Hub does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which QAI Hub can then upload, compile, and convert to its QNN DLC format in the cloud.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=13`.
4. Applies **ONNX graph surgery** to eliminate all mixed-range intermediate tensors and produce two independent output branches.

The model decodes the grid offsets inside the graph and returns two outputs:

| Output   | Shape          | Range  | Description                              |
|----------|----------------|--------|------------------------------------------|
| `bboxes` | `[1, 8400, 4]` | 0–640  | Decoded bbox coords `[cx, cy, w, h]`     |
| `scores` | `[1, 8400, 81]`| 0–1    | Objectness + 80 class probabilities      |

where `8400 = 80×80 + 40×40 + 20×20` anchor-free grid cells.

The bbox format is `[cx, cy, w, h]` in the resized 640×640 input coordinate space. When using the model, confidence filtering, `cxcywh → xyxy` conversion, scaling back to the original image, and NMS are still required.

In [ ]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../models/qaihub", exist_ok=True)

if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

checkpoint = torch.load(
    "../models/LibreYOLOXs.pt",
    map_location="cpu"
)

model = LibreYOLOXModel(config="s", nb_classes=80)
model.load_state_dict(checkpoint["model"], strict=True)

model.eval()
model.head.export = True


def _split_onnx_for_dsp(path):
    """Restructure the ONNX graph to eliminate mixed-range intermediate tensors.

    The original YOLOX export produces per-scale Concat nodes that mix decoded
    bbox coordinates (range 0-640) with sigmoid scores (range 0-1) in a single
    [1, n, 85] tensor.  With INT8 per-tensor quantization, the scale is set to
    cover 0-640, which maps all 0-1 score values to INT8 zero on DSP.

    This function removes those mixed-range Concat nodes and wires the already-
    existing [1, n, 81] score tensors directly into a separate cross-scale
    Concat, producing two independent output branches with their own INT8 scales:

        bboxes  [1, 8400, 4]  -- range 0-640   -> INT8 scale ~2.5 / step
        scores  [1, 8400, 81] -- range 0-1      -> INT8 scale ~0.004 / step

    Every new node is given an explicit name so that SNPE/QAIRT DLC layer
    names are predictable (e.g. setOutputLayers({"bboxes", "scores"})).

    The function is idempotent: if no mixed-range nodes are found, it prints a
    message and returns without modifying the file.
    """
    model = onnx.load(path)
    model = onnx.shape_inference.infer_shapes(model)
    graph = model.graph

    shape_map = {
        vi.name: [d.dim_value for d in vi.type.tensor_type.shape.dim]
        for vi in list(graph.value_info) + list(graph.input) + list(graph.output)
        if vi.type.HasField("tensor_type") and vi.type.tensor_type.shape
    }

    SCALE_NS = {6400, 1600, 400}
    per_scale, final_cat = [], None

    for node in graph.node:
        if node.op_type != "Concat":
            continue
        s = shape_map.get(node.output[0], [])
        ax = next((a.i for a in node.attribute if a.name == "axis"), None)
        if len(s) == 3 and s[0] == 1 and s[2] == 85 and s[1] in SCALE_NS and ax == -1:
            per_scale.append(node)
        elif s == [1, 8400, 85] and ax == 1:
            final_cat = node

    if not per_scale:
        print("ONNX graph already restructured, skipping.")
        return

    assert len(per_scale) == 3 and final_cat is not None, \
        "Unexpected graph structure — cannot apply surgery."

    old_to_new, new_nodes = {}, []

    for node in per_scale:
        n_k = shape_map[node.output[0]][1]
        inps = list(node.input)
        bbox_inps  = [i for i in inps if shape_map.get(i, [])[-1:] == [2]]
        score_inps = [i for i in inps if shape_map.get(i, [])[-1:] == [81]]
        bbox_name = f"_split_bboxes_{n_k}"
        new_nodes.append(helper.make_node(
            "Concat", inputs=bbox_inps, outputs=[bbox_name],
            name=bbox_name, axis=-1
        ))
        old_to_new[node.output[0]] = (bbox_name, score_inps[0])

    all_bboxes = [old_to_new[i][0] for i in final_cat.input]
    all_scores = [old_to_new[i][1] for i in final_cat.input]
    new_nodes.append(helper.make_node(
        "Concat", inputs=all_bboxes, outputs=["bboxes"],
        name="bboxes", axis=1
    ))
    new_nodes.append(helper.make_node(
        "Concat", inputs=all_scores, outputs=["scores"],
        name="scores", axis=1
    ))

    final_tensor = final_cat.output[0]
    remove_ids = {id(n) for n in per_scale} | {id(final_cat)}
    for node in graph.node:
        if final_tensor in list(node.input):
            remove_ids.add(id(node))

    new_graph_nodes = [n for n in graph.node if id(n) not in remove_ids] + new_nodes
    del graph.node[:]
    graph.node.extend(new_graph_nodes)

    while graph.output:
        graph.output.pop()
    graph.output.extend([
        helper.make_tensor_value_info("bboxes", TensorProto.FLOAT, [1, 8400, 4]),
        helper.make_tensor_value_info("scores", TensorProto.FLOAT, [1, 8400, 81]),
    ])

    model = onnx.shape_inference.infer_shapes(model)
    onnx.checker.check_model(model)
    onnx.save(model, path)
    print("ONNX graph restructured: bboxes and scores are now separate branches.")

dummy = torch.randn(1, 3, 640, 640)
onnx_path = "../models/LibreYOLOXs.onnx"

if not os.path.exists(onnx_path):
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        opset_version=13,
        dynamo=False,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["detections"]
    )

    print(f"Exported decoded ONNX to: {onnx_path}.")

_split_onnx_for_dsp(onnx_path)


### Validating the ONNX output shapes

This cell checks that the exported ONNX has exactly two outputs — `bboxes` with shape `[1, 8400, 4]` and `scores` with shape `[1, 8400, 81]`. If you still see a single `detections` output with 85 columns, delete the ONNX file and re-run the previous cell so that `_split_onnx_for_dsp()` can restructure the graph.

In [ ]:
onnx_model = onnx.load("../models/LibreYOLOXs.onnx")
onnx.checker.check_model(onnx_model)

print("Inputs:")
for tensor in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

assert len(onnx_model.graph.output) == 2, "Expected exactly two outputs (bboxes, scores)."
assert "bboxes" in output_shapes, "Expected output named `bboxes`."
assert "scores" in output_shapes, "Expected output named `scores`."
assert output_shapes["bboxes"] == [1, 8400, 4], f"Unexpected bboxes shape: {output_shapes['bboxes']}"
assert output_shapes["scores"] == [1, 8400, 81], f"Unexpected scores shape: {output_shapes['scores']}"

print("ONNX export is correct: bboxes [1, 8400, 4] and scores [1, 8400, 81].")

## Setting Up QAI Hub

### Get Your QAI Hub API Token and Configure it

Before configuring, you need to get your QAI Hub API Token:

1. **Sign into AI Hub**: Go to https://app.aihub.qualcomm.com/
2. **Navigate to Settings**: Click on "Your ID" → "Settings" → "API Token"
3. **Environment Variable**: Create the `.env` file to configure the development environment
4. **Copy Your Token**: Copy the QAI Hub API Token to the variable `QAI_HUB_API_TOKEN`

⚠️ **Security Note**: Keep your API token private. Don't commit it to version control.

In [ ]:
config = dotenv_values(".env")
api_token = config["QAI_HUB_API_TOKEN"]

_ = subprocess.run(
    [
        "qai-hub",
        "configure",
        "--api_token",
        api_token
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

### Listing Available QAI Hub Devices

Before targeting a specific device for compilation and profiling, you can query the full list of devices supported by QAI Hub. The `qai-hub list-devices` command returns all currently available Snapdragon-powered devices — including smartphones, tablets, and compute platforms — that are accessible through QAI Hub's cloud infrastructure.

This helps identify supported device names to pass to `hub.Device()` and `client.submit_compile_job()` in subsequent steps.

In [ ]:
!qai-hub list-devices

### Querying a Specific Device

Once you have identified a target device, you can query its detailed specification — including supported compute units (CPU, GPU, HTP/DSP) and chipset information — using the `--device` flag. This confirms that the chosen device is available and reveals the hardware backends accessible for model execution.

The **Samsung Galaxy A73 5G**, powered by the Snapdragon 778G chipset, supports CPU, GPU, and HTP (Hexagon Tensor Processor) backends, making it a suitable target for both FP32 and INT8 inference profiling.

In [ ]:
!qai-hub list-devices --device "Samsung Galaxy A73 5G"

## Converting the Model to QAI Hub DLC Format

QAI Hub compiles models to Qualcomm's **QNN DLC (Deep Learning Container)** format in the cloud, targeting a specific Snapdragon device. Unlike local SDK tools such as `snpe-onnx-to-dlc` or `qairt-converter`, the entire compilation pipeline runs remotely on Qualcomm's infrastructure — no local SDK installation is required.

The cell below initializes the QAI Hub client, selects the target device, and defines the output paths for the FP32 and INT8 DLC files.

In [ ]:
client = hub.Client()

DEVICE = hub.Device("Samsung Galaxy A73 5G")

FP32_ONNX = "../models/LibreYOLOXs.onnx"
FP32_DLC = "../models/qaihub/LibreYOLOXs_fp32.dlc"
INT8_DLC = "../models/qaihub/LibreYOLOXs_int8.dlc"

### Compiling the FP32 Model

The ONNX model is first uploaded to QAI Hub via `client.upload_model()`, then compiled to a **floating-point QNN DLC** by submitting a compile job. The job runs on Qualcomm's cloud infrastructure and targets the chosen Snapdragon device.

Compilation options specify:
- `--target_runtime qnn_dlc` — produces a QNN-compatible DLC for on-device inference.
- `--output_names bboxes,scores` — maps the restructured ONNX outputs to named outputs in the compiled DLC.

Once the job completes, the compiled FP32 DLC is downloaded locally for profiling.

In [ ]:
fp32_onnx_model = client.upload_model(FP32_ONNX)

compile_options = [
    "--target_runtime qnn_dlc",     # Qualcomm runtime
    "--output_names bboxes,scores"  # Custom output names
]
options_str = " ".join(compile_options)

compile_fp32_job = client.submit_compile_job(
    model=fp32_onnx_model,
    device=DEVICE,
    options=options_str,
    input_specs={
        "images": (1, 3, 640, 640),
    },
)
compile_fp32_job.wait()

fp32_dlc_model = compile_fp32_job.get_target_model()
fp32_dlc_model.download(FP32_DLC)

### Profiling the FP32 Model

QAI Hub measures actual on-device inference latency by submitting a profile job that runs the compiled model on the target hardware. For the FP32 model, profiling is limited to `cpu` and `gpu` compute units — the HTP (Hexagon Tensor Processor) backend requires INT8 precision and is not compatible with floating-point DLCs.

The returned job object includes detailed per-layer and end-to-end timing results, which can be inspected directly in the notebook.

In [ ]:
profile_fp32_job = client.submit_profile_job(
    model=fp32_dlc_model,
    device=DEVICE,
    options="--compute_unit cpu,gpu",
)
profile_fp32_job.wait()
profile_fp32_job

### INT8 Post-Training Quantization and Compilation

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. QAI Hub performs quantization in the cloud via `client.submit_quantize_job()`, using the calibration `.raw` files loaded as NumPy arrays.

The calibration inputs are passed as a dictionary of NumPy arrays keyed by the input tensor name `"images"`. After quantization, the resulting model is compiled to a QNN DLC with INT8 precision and downloaded locally.

Compilation options specify:
- `--quantize_full_type int8` — applies full INT8 quantization to all weights and activations.
- `--quantize_io` — quantizes the model's input and output tensors.
- `--target_runtime qnn_dlc` and `--output_names bboxes,scores` — same as the FP32 compilation step.

In [ ]:
raw_files = open("qaihub/calib/filenames.txt").read().splitlines()

calibration_inputs = {
    "images": [
        np.fromfile(path, dtype=np.float32).reshape(1, 3, 640, 640)
        for path in raw_files
    ]
}

quantize_job = client.submit_quantize_job(
    model=fp32_onnx_model,
    calibration_data=calibration_inputs,
)
quantized_model = quantize_job.get_target_model()

compile_options = [
    "--target_runtime qnn_dlc",     # Qualcomm runtime
    "--quantize_full_type int8",    # 8-bit quantization  
    "--quantize_io",                # Quantize I/O
    "--output_names bboxes,scores"  # Custom output names
]
options_str = " ".join(compile_options)

compile_int8_job = client.submit_compile_job(
    model=quantized_model,
    device=DEVICE,
    options=options_str,
    input_specs={
        "images": (1, 3, 640, 640),
    },
)
compile_int8_job.wait()

int8_dlc_model = compile_int8_job.get_target_model()
int8_dlc_model.download(INT8_DLC)

### Profiling the INT8 Model

The INT8 DLC is profiled across all available compute units — including the **HTP (Hexagon Tensor Processor / DSP)** — using `--compute_unit all`. The HTP backend is specifically designed for INT8 inference and delivers the highest throughput and lowest power consumption on Qualcomm® Snapdragon chipsets.

Comparing the FP32 and INT8 profiling results shows the latency reduction and speedup achieved through quantization on the target device.

In [ ]:
profile_int8_job = client.submit_profile_job(
    model=int8_dlc_model,
    device=DEVICE,
    options="--compute_unit all",
)
profile_int8_job.wait()
profile_int8_job

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.